# Phase 3 — Size Recommender
**ADSP 31017 Machine Learning I — Winter 2026**  
**Authors:** Harleen Kaur Buttar, Skylar Liu, Dora Jiayue Li

Builds a collaborative-filtering size recommender on top of the fit probabilities saved in Phase 2.

**Pipeline:**
1. Load per-model fit probability CSVs → soft ensemble P(Fit)
2. Merge ensemble probabilities onto RentTheRunway transaction data
3. Build sparse user × (item_id, size) matrix valued by P(Fit)
4. Cosine-similarity collaborative filtering → recommend size per user–item pair
5. Evaluate hit-rate (warm-start) and cold-start via nearest-measurement neighbour
6. Cross-retailer generalisation: train on ModCloth, evaluate on RentTheRunway

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

PROCESSED_DIR = Path('Data/Processed')

# Must match the prefix of your saved Phase 2 files exactly
# e.g. Linear_SVM_save_fit_probabilies.csv
MODEL_NAMES = [
    'Linear_SVM',
    'Kernel_SVM',
    'CART',
    'Random_Forest',
    'Bagging',
    'KNN',
    'Naive_Bayes',
]
PROBA_COLS = ['proba_Small', 'proba_Fit', 'proba_Large']

## 1. Load Per-Model Probabilities → Soft Ensemble

In [ ]:
def load_ensemble_proba(model_names=MODEL_NAMES, weight_dict=None):
    """
    Average P(Small/Fit/Large) across all saved {model}_save_fit_probabilies.csv files.
    weight_dict: optional dict[model_name, float] — uniform if None.
    """
    weighted_sum = None
    total_weight = 0.0
    true_labels  = None

    for name in model_names:
        path = PROCESSED_DIR / f'{name}_save_fit_probabilies.csv'
        if not path.exists():
            print(f'  [WARN] {path.name} not found — skipping.')
            continue

        df = pd.read_csv(path, index_col=0)
        w  = (weight_dict or {}).get(name, 1.0)
        cols = [c for c in PROBA_COLS if c in df.columns]

        if weighted_sum is None:
            weighted_sum = df[cols].values * w
            true_labels  = df.get('true_label')
        else:
            weighted_sum += df[cols].values * w

        total_weight += w
        print(f'  Loaded {path.name}  (weight={w})')

    if weighted_sum is None:
        raise FileNotFoundError('No probability files found in Data/Processed/')

    ensemble = pd.DataFrame(weighted_sum / total_weight, columns=PROBA_COLS)
    ensemble['pred_label'] = ensemble[PROBA_COLS].idxmax(axis=1).str.replace('proba_', '')
    if true_labels is not None:
        ensemble['true_label'] = true_labels.values

    print(f'\n  Ensemble built from {int(total_weight)} models — {len(ensemble):,} rows')
    return ensemble

In [ ]:
ensemble_df = load_ensemble_proba(MODEL_NAMES)
ensemble_df.head()

### Ensemble prediction distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ensemble_df['pred_label'].value_counts().plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='black'
)
axes[0].set_title('Ensemble Predicted Label Distribution')
axes[0].set_xlabel('Predicted Fit')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

ensemble_df[PROBA_COLS].mean().plot(
    kind='bar', ax=axes[1],
    color=['#e07b54', '#5b9e6d', '#5b7fba'], edgecolor='black'
)
axes[1].set_title('Mean Ensemble Probability per Class')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Mean Probability')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 2. Load Transaction Data & Merge Ensemble Probabilities

In [ ]:
def load_data(retailer='renttherunway', ensemble_df=None):
    """
    Load cleaned transaction CSV and merge ensemble P(Fit) by index.
    Falls back to binary fit-label proxy if no ensemble provided.
    """
    path = PROCESSED_DIR / f'{retailer}_clean.csv'
    df   = pd.read_csv(path)

    rename = {'bust size': 'bust_size', 'bra size': 'bust_size'}
    df = df.rename(columns={k: v for k, v in rename.items() if k in df.columns})

    if ensemble_df is not None:
        df = df.join(ensemble_df[PROBA_COLS], how='left')
        print(f'  Merged ensemble probs into {retailer} ({df.shape[0]:,} rows).')
    else:
        df['proba_Fit'] = (df['fit'] == 'Fit').astype(float)
        print(f'  {retailer}: {df.shape[0]:,} rows (binary proba_Fit fallback).')

    return df

In [ ]:
df_rtr = load_data('renttherunway', ensemble_df=ensemble_df)
print(df_rtr.shape)
df_rtr[['user_id', 'item_id', 'size', 'fit', 'proba_Small', 'proba_Fit', 'proba_Large']].head()

## 3. Build Sparse User–Item Matrix

In [ ]:
def build_user_item_matrix(df, value_col='proba_Fit'):
    """
    csr_matrix  rows=users  cols=(item_id + '__' + size)  vals=P(Fit)
    Returns: matrix, user_le, item_le
    """
    df = df.dropna(subset=['user_id', 'item_id', 'size', value_col]).copy()
    df['item_key'] = df['item_id'].astype(str) + '__' + df['size'].astype(str)

    user_le = LabelEncoder().fit(df['user_id'])
    item_le = LabelEncoder().fit(df['item_key'])

    row = user_le.transform(df['user_id'])
    col = item_le.transform(df['item_key'])
    val = df[value_col].values.astype(float)

    n_u, n_i = len(user_le.classes_), len(item_le.classes_)
    mat = csr_matrix((val, (row, col)), shape=(n_u, n_i))

    print(f'  Matrix: {n_u:,} users x {n_i:,} (item, size) pairs')
    print(f'  Sparsity: {1 - mat.nnz / (n_u * n_i):.4%}')
    return mat, user_le, item_le

## 4. Collaborative-Filtering Recommender

In [ ]:
class SizeRecommender:
    """
    Cosine-similarity CF recommender.
    Warm-start : known user_id -> use their matrix row directly.
    Cold-start : measurements dict -> proxy via nearest-measurement training user.
    """

    def __init__(self, k=20):
        self.k        = k
        self.matrix   = None
        self.user_le  = None
        self.item_le  = None
        self.df_train = None

    def fit(self, df, value_col='proba_Fit'):
        self.df_train = df.copy()
        self.matrix, self.user_le, self.item_le = build_user_item_matrix(df, value_col)
        return self

    def _row(self, user_id):
        if user_id not in set(self.user_le.classes_):
            return None
        return self.matrix[self.user_le.transform([user_id])[0]]

    def _cold_start_row(self, measurements):
        """Find training user with closest body measurements (L2 distance)."""
        meas_cols = [c for c in ['height', 'weight', 'age', 'bust_size']
                     if c in self.df_train.columns]
        query = np.array([measurements.get(c, np.nan) for c in meas_cols], dtype=float)
        valid = ~np.isnan(query)
        if valid.sum() == 0:
            return None
        train_meas = self.df_train.groupby('user_id')[meas_cols].first().dropna()
        dists = np.linalg.norm(train_meas.iloc[:, valid].values - query[valid], axis=1)
        nearest = train_meas.index[np.argmin(dists)]
        return self._row(nearest)

    def recommend(self, item_id, user_id=None, measurements=None):
        """
        Returns DataFrame [size, avg_proba_Fit, n_neighbours] sorted descending.
        Provide user_id (warm-start) OR measurements dict (cold-start).
        """
        if user_id is not None:
            query_vec = self._row(user_id)
        elif measurements is not None:
            query_vec = self._cold_start_row(measurements)
        else:
            raise ValueError('Provide user_id or measurements.')

        if query_vec is None:
            return pd.DataFrame(columns=['size', 'avg_proba_Fit', 'n_neighbours'])

        item_str  = str(item_id)
        item_cols = {k: i for i, k in enumerate(self.item_le.classes_)
                     if k.startswith(item_str + '__')}
        if not item_cols:
            return pd.DataFrame(columns=['size', 'avg_proba_Fit', 'n_neighbours'])

        sim = cosine_similarity(query_vec, self.matrix).flatten()

        if user_id is not None and user_id in set(self.user_le.classes_):
            sim[self.user_le.transform([user_id])[0]] = -1.0

        records = []
        for item_key, col_idx in item_cols.items():
            size     = item_key.split('__', 1)[1]
            col_vals = self.matrix[:, col_idx].toarray().flatten()
            interacted = np.where(col_vals > 0)[0]
            if len(interacted) == 0:
                continue
            top_k = interacted[np.argsort(sim[interacted])[::-1][: self.k]]
            records.append({
                'size':          size,
                'avg_proba_Fit': round(col_vals[top_k].mean(), 4),
                'n_neighbours':  len(top_k),
            })

        return (
            pd.DataFrame(records)
            .sort_values('avg_proba_Fit', ascending=False)
            .reset_index(drop=True)
        )

## 5. Train / Test Split & Fit Recommender

In [ ]:
# User-level split to prevent leakage
train_users, test_users = train_test_split(
    df_rtr['user_id'].unique(), test_size=0.2, random_state=42
)
df_train = df_rtr[df_rtr['user_id'].isin(train_users)]
df_test  = df_rtr[df_rtr['user_id'].isin(test_users)]

print(f'Train: {len(df_train):,} rows  |  Test: {len(df_test):,} rows')

rec = SizeRecommender(k=20).fit(df_train)

## 6. Evaluation — Hit-Rate@N

In [ ]:
def evaluate_recommender(rec, df_test, top_n=1):
    """
    Hit-rate@N: is the true worn size in the top-N recommendations?
    Also reports fit_only_hit_rate for rows where actual fit == 'Fit'.
    """
    hits = fit_hits = fit_total = total = 0

    for _, row in df_test.iterrows():
        try:
            recs = rec.recommend(item_id=row['item_id'], user_id=row.get('user_id'))
        except Exception:
            continue
        if recs.empty:
            continue

        top_sizes = recs.head(top_n)['size'].tolist()
        if str(row['size']) in top_sizes:
            hits += 1
            if row.get('fit') == 'Fit':
                fit_hits += 1
        if row.get('fit') == 'Fit':
            fit_total += 1
        total += 1

    return {
        f'hit_rate_top{top_n}':          round(hits / total, 4)         if total     else 0,
        f'fit_only_hit_rate_top{top_n}': round(fit_hits / fit_total, 4) if fit_total else 0,
        'total_evaluated': total,
    }

In [ ]:
results = {}
results.update(evaluate_recommender(rec, df_test, top_n=1))
results.update(evaluate_recommender(rec, df_test, top_n=3))

print('Within-retailer evaluation (RentTheRunway):')
pd.Series(results)

### Visualise hit-rate results

In [ ]:
metrics = {
    'Hit@1 (all)':      results['hit_rate_top1'],
    'Hit@1 (Fit only)': results['fit_only_hit_rate_top1'],
    'Hit@3 (all)':      results['hit_rate_top3'],
    'Hit@3 (Fit only)': results['fit_only_hit_rate_top3'],
}

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(
    metrics.keys(), metrics.values(),
    color=['#5b7fba', '#5b9e6d', '#5b7fba', '#5b9e6d'],
    edgecolor='black', alpha=0.85
)
ax.bar_label(bars, fmt='%.3f', padding=3)
ax.set_ylim(0, 1.0)
ax.set_title('Recommender Hit-Rate (RentTheRunway)')
ax.set_ylabel('Rate')
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

## 7. Warm-Start Demo

In [ ]:
sample_row  = df_test.dropna(subset=['item_id', 'size']).iloc[0]
sample_user = sample_row['user_id']
sample_item = sample_row['item_id']

recs = rec.recommend(item_id=sample_item, user_id=sample_user)

print(f'User: {sample_user}  |  Item: {sample_item}')
print(f'Actual size worn: {sample_row["size"]}  |  Actual fit: {sample_row["fit"]}')
print()
recs

## 8. Cold-Start Demo (New Customer — No Purchase History)

In [ ]:
new_customer = {'height': 65, 'weight': 130, 'age': 28, 'bust_size': 36}
popular_item = df_train['item_id'].value_counts().index[0]

cold_recs = rec.recommend(item_id=popular_item, measurements=new_customer)

print(f'Cold-start recommendations for item {popular_item}:')
cold_recs

## 9. Cross-Retailer Generalisation (ModCloth → RentTheRunway)

In [ ]:
# Train on ModCloth (binary fit labels), evaluate on held-out RTR transactions
df_modcloth = load_data('modcloth')

_, df_rtr_hold = train_test_split(
    df_rtr, test_size=0.2, random_state=42, stratify=df_rtr['fit']
)

rec_cross = SizeRecommender(k=20).fit(df_modcloth)

cross_results = {}
cross_results.update(evaluate_recommender(rec_cross, df_rtr_hold, top_n=1))
cross_results.update(evaluate_recommender(rec_cross, df_rtr_hold, top_n=3))

print('Cross-retailer results (ModCloth -> RentTheRunway):')
pd.Series(cross_results)

### Within-retailer vs Cross-retailer comparison

In [ ]:
compare = pd.DataFrame({
    'Within-Retailer (RTR)': [
        results['hit_rate_top1'],
        results['hit_rate_top3'],
        results['fit_only_hit_rate_top1'],
        results['fit_only_hit_rate_top3'],
    ],
    'Cross-Retailer (MC->RTR)': [
        cross_results['hit_rate_top1'],
        cross_results['hit_rate_top3'],
        cross_results['fit_only_hit_rate_top1'],
        cross_results['fit_only_hit_rate_top3'],
    ],
}, index=['Hit@1 (all)', 'Hit@3 (all)', 'Hit@1 (Fit only)', 'Hit@3 (Fit only)'])

compare.plot(
    kind='bar', figsize=(9, 5),
    color=['#5b7fba', '#e07b54'], edgecolor='black', alpha=0.85
)
plt.title('Within-Retailer vs Cross-Retailer Hit-Rate')
plt.ylabel('Rate')
plt.ylim(0, 1.0)
plt.xticks(rotation=15)
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

compare